# 01b — Extract multilingual text embeddings for items_phase_2

Mirrors the recipe used for `items_train` + `items_phase_1` in
[../phase1/07_compute_fashion_clip_embeddings.ipynb](../phase1/07_compute_fashion_clip_embeddings.ipynb)
(cell 4) — same multilingual SentenceTransformer model
(`sentence-transformers/clip-ViT-B-32-multilingual-v1`, 512-d, already L2-normalised).

**Strategy**: append phase_2 itemIds to the *existing*
`artifacts/embeddings/text_multilingual.pt` dictionary. Resume-safe — running
this notebook before `items_phase_2.csv` is released does nothing; running it
after only encodes the new ~200k items.

This way the rest of the pipeline (clustering, training) keeps loading a single
`text_multilingual.pt` that contains every item across all phases.


In [1]:
import sys, os
sys.path.insert(0, '/Users/matouskovar/FIT/adm-sp/src')

REPO_ROOT          = '/Users/matouskovar/FIT/adm-sp'
DATA_DIR           = f'{REPO_ROOT}/data'
ARTIFACTS_DIR      = f'{REPO_ROOT}/artifacts'
TEXT_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/text_multilingual.pt'

import torch
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: mps


In [2]:
# Dependencies (run once)
# %pip install sentence-transformers pandas tqdm torch


In [3]:
import pandas as pd
from tqdm.auto import tqdm

p2_path = f'{DATA_DIR}/items_phase_2.csv'
if not os.path.exists(p2_path):
    raise FileNotFoundError(
        f'{p2_path} not found. items_phase_2.csv has not been released yet — '
        f'come back when it is. (This notebook is a no-op until then.)'
    )

df_p2 = pd.read_csv(p2_path)
df_p2['itemId_str']  = df_p2['itemId'].astype(str)
df_p2['title']       = df_p2['title'].fillna('')
df_p2['description'] = df_p2['description'].fillna('')
df_p2['text']        = (df_p2['title'] + ' ' + df_p2['description']).str.strip()

print(f'items_phase_2 rows: {len(df_p2):,}')


items_phase_2 rows: 197,837


In [4]:
# Load the existing text-embedding dict (train + phase1) and figure out
# which phase_2 items still need encoding.
if os.path.exists(TEXT_EMB_PATH):
    text_embeddings = torch.load(TEXT_EMB_PATH, map_location='cpu', weights_only=False)
    print(f'Existing dict size: {len(text_embeddings):,}')
else:
    text_embeddings = {}
    print('No existing text_multilingual.pt — starting from scratch.')

p2_ids       = df_p2['itemId_str'].tolist()
remaining    = [iid for iid in p2_ids if iid not in text_embeddings]
print(f'Phase 2 items still to encode: {len(remaining):,}')


Existing dict size: 1,128,069
Phase 2 items still to encode: 197,837


In [5]:
from sentence_transformers import SentenceTransformer

MODEL_ID = 'sentence-transformers/clip-ViT-B-32-multilingual-v1'
print(f'Loading {MODEL_ID}...')
text_model = SentenceTransformer(MODEL_ID, device=str(device))
print(f'  Embedding dim: {text_model.get_sentence_embedding_dimension()}')   # 512


Loading sentence-transformers/clip-ViT-B-32-multilingual-v1...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  Embedding dim: 512


In [6]:
TEXT_BATCH_SIZE = 512   # ST handles internal batching; we chunk for checkpointing
texts_map = df_p2.set_index('itemId_str')['text'].to_dict()

processed_since_save = 0
CHECKPOINT_EVERY     = 50_000

for batch_start in tqdm(range(0, len(remaining), TEXT_BATCH_SIZE), desc='Text embeddings'):
    batch_ids   = remaining[batch_start : batch_start + TEXT_BATCH_SIZE]
    batch_texts = [texts_map.get(iid, '') for iid in batch_ids]

    with torch.no_grad():
        feats = text_model.encode(
            batch_texts,
            batch_size=TEXT_BATCH_SIZE,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).cpu()    # (B, 512)

    for iid, feat in zip(batch_ids, feats):
        text_embeddings[iid] = feat

    processed_since_save += len(batch_ids)
    if processed_since_save >= CHECKPOINT_EVERY:
        torch.save(text_embeddings, TEXT_EMB_PATH)
        processed_since_save = 0

torch.save(text_embeddings, TEXT_EMB_PATH)
print(f'\nSaved {len(text_embeddings):,} total embeddings → {TEXT_EMB_PATH}')


Text embeddings:   0%|          | 0/387 [00:00<?, ?it/s]


Saved 1,325,906 total embeddings → /Users/matouskovar/FIT/adm-sp/artifacts/embeddings/text_multilingual.pt


## Sanity check

Verify a few phase_2 items got encoded with the right shape and that they
respect the L2-norm convention (cosine = dot product on these vectors).


In [7]:
import torch.nn.functional as F

sample_ids = df_p2['itemId_str'].head(5).tolist()
for iid in sample_ids:
    v = text_embeddings.get(iid)
    if v is None:
        print(f'  {iid}: MISSING')
        continue
    norm = v.norm().item()
    print(f'  {iid}: shape={tuple(v.shape)}  ||v||={norm:.4f}')

# Quick same-vs-different sanity (within-language is fine here, just shape check)
if len(sample_ids) >= 2:
    a, b = sample_ids[0], sample_ids[1]
    sim = F.cosine_similarity(text_embeddings[a].unsqueeze(0), text_embeddings[b].unsqueeze(0)).item()
    print(f'\nCos sim {a} vs {b}: {sim:.3f}')


  500711: shape=(512,)  ||v||=1.0000
  461542: shape=(512,)  ||v||=1.0000
  45654: shape=(512,)  ||v||=1.0000
  974839: shape=(512,)  ||v||=1.0000
  922483: shape=(512,)  ||v||=1.0000

Cos sim 500711 vs 461542: 0.969
